In [1]:
import re
import string
import emoji

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from spellchecker import SpellChecker

In [2]:
import pandas as pd 
import numpy as np


In [3]:
df= pd.read_csv(r"C:\Users\himan\Education1\Projects\NLP\data\movie.rating.csv")
df.head()

,Unnamed: 0,id,title,overview,release_date,popularity,vote_average,vote_count
0,0,14869,G.I. Joe: The Rise of Cobra,From the Egyptian desert to deep below the pol...,2009-08-03,9.5459,5.783,5018
1,1,4561,The Story of O,"The beautiful O is taken by her boyfriend, Ren...",1975-08-27,10.5329,5.497,189
2,2,41446,Scream 4,Fifteen years after the original Woodsboro mur...,2011-04-13,10.3303,6.482,4290
3,3,1417285,The Stain,"When Malee, a young woman with a dark past, is...",2026-02-26,10.3305,4.700,3
4,4,263354,The Secret,"Lucia (Natassja Kinski) is a volatile, excitea...",1990-02-16,9.7878,4.000,2


In [5]:
df= df.drop("Unnamed: 0", axis= 1)
df.head()

,id,title,overview,release_date,popularity,vote_average,vote_count
0,14869,G.I. Joe: The Rise of Cobra,From the Egyptian desert to deep below the pol...,2009-08-03,9.5459,5.783,5018
1,4561,The Story of O,"The beautiful O is taken by her boyfriend, Ren...",1975-08-27,10.5329,5.497,189
2,41446,Scream 4,Fifteen years after the original Woodsboro mur...,2011-04-13,10.3303,6.482,4290
3,1417285,The Stain,"When Malee, a young woman with a dark past, is...",2026-02-26,10.3305,4.700,3
4,263354,The Secret,"Lucia (Natassja Kinski) is a volatile, excitea...",1990-02-16,9.7878,4.000,2


# Applying Lowercase

In [7]:
df["overview"]= df["overview"].str.lower()
df["overview"]

0       from the egyptian desert to deep below the pol...
1       the beautiful o is taken by her boyfriend, ren...
2       fifteen years after the original woodsboro mur...
3       when malee, a young woman with a dark past, is...
4       lucia (natassja kinski) is a volatile, excitea...
                              ...                        
9975    beneath anna poliatova's striking beauty lies ...
9976    fresh to las vegas with no connections, nomi m...
9977    after twenty years away, odysseus washes up on...
9978    madeline is married to ernest, who was once he...
9979    a tenacious attorney uncovers a dark secret th...
Name: overview, Length: 9980, dtype: str

In [8]:
df.head()

,id,title,overview,release_date,popularity,vote_average,vote_count
0,14869,G.I. Joe: The Rise of Cobra,from the egyptian desert to deep below the pol...,2009-08-03,9.5459,5.783,5018
1,4561,The Story of O,"the beautiful o is taken by her boyfriend, ren...",1975-08-27,10.5329,5.497,189
2,41446,Scream 4,fifteen years after the original woodsboro mur...,2011-04-13,10.3303,6.482,4290
3,1417285,The Stain,"when malee, a young woman with a dark past, is...",2026-02-26,10.3305,4.700,3
4,263354,The Secret,"lucia (natassja kinski) is a volatile, excitea...",1990-02-16,9.7878,4.000,2


In [9]:
df["title"]= df["title"].str.lower()

In [12]:
df["overview"] = df["overview"].fillna("")

# Removing HTML Tags, Puncuations, Urls

In [13]:
def remove_html_tags(text):
    pattern = re.compile("<.*?>")
    return pattern.sub("", text)

In [14]:
df["overview"]= df["overview"].apply(remove_html_tags)

In [15]:
def remove_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", "", text)

In [16]:
df["overview"]= df["overview"].apply(remove_urls)

In [17]:
def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

In [18]:
df["overview"]= df["overview"].apply(remove_punctuation)

# Chatword Treatment

In [19]:
chat_words = {
    "A3": "Anytime, Anywhere, Anyplace",
    "ADIH": "Another Day In Hell",
    "AFK": "Away From Keyboard",
    "AFAIK": "As Far As I Know",
    "ASAP": "As Soon As Possible",
    "ASL": "Age, Sex, Location",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "BAE": "Before Anyone Else",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRUH": "Bro",
    "BRT": "Be Right There",
    "BSAAW": "Big Smile And A Wink",
    "BTW": "By The Way",
    "BWL": "Bursting With Laughter",
    "CSL": "Can't Stop Laughing",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "DM": "Direct Message",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FIMH": "Forever In My Heart",
    "FOMO": "Fear Of Missing Out",
    "FR": "For Real",
    "FWIW": "For What It's Worth",
    "FYP": "For You Page",
    "FYI": "For Your Information",
    "G9": "Genius",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GMTA": "Great Minds Think Alike",
    "GN": "Good Night",
    "GOAT": "Greatest Of All Time",
    "GR8": "Great",
    "HBD": "Happy Birthday",
    "IC": "I See",
    "ICQ": "I Seek You",
    "IDC": "I Don't Care",
    "IDK": "I Don't Know",
    "IFYP": "I Feel Your Pain",
    "ILU": "I Love You",
    "ILY": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMU": "I Miss You",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "IYKYK": "If You Know, You Know",
    "JK": "Just Kidding",
    "KISS": "Keep It Simple, Stupid",
    "L": "Loss",
    "L8R": "Later",
    "LDR": "Long Distance Relationship",
    "LMK": "Let Me Know",
    "LMAO": "Laughing My A** Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "M8": "Mate",
    "MFW": "My Face When",
    "MID": "Mediocre",
    "MRW": "My Reaction When",
    "MTE": "My Thoughts Exactly",
    "NVM": "Never Mind",
    "NRN": "No Reply Necessary",
    "NPC": "Non-Player Character",
    "OIC": "Oh I See",
    "OP": "Overpowered",
    "PITA": "Pain In The A**",
    "POV": "Point Of View",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A** Off",
    "RN": "Right Now",
    "SK8": "Skate",
    "STATS": "Your Sex And Age",
    "SUS": "Suspicious",
    "TBH": "To Be Honest",
    "TFW": "That Feeling When",
    "THX": "Thank You",
    "TIME": "Tears In My Eyes",
    "TLDR": "Too Long, Didn't Read",
    "TNTL": "Trying Not To Laugh",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "W": "Win",
    "W8": "Wait",
    "WB": "Welcome Back",
    "WTF": "What The F**k",
    "WTG": "Way To Go",
    "WUF": "Where Are You From",
    "WYD": "What You Doing",
    "WYWH": "Wish You Were Here",
    "ZZZ": "Sleeping, Bored, Tired"
}

In [20]:
chat_words = {key.lower(): value.lower() for key, value in chat_words.items()}

In [21]:
def chat_word_treatment(text):

    words = text.split()

    new_words = []

    for word in words:
        if word.lower() in chat_words:
            new_words.append(chat_words[word.lower()])
        else:
            new_words.append(word)

    return " ".join(new_words)

In [23]:
df["overview"]= df["overview"].apply(chat_word_treatment)

In [24]:
import tqdm

# Handle Emojis

In [26]:
def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")

In [27]:
df["overview"]= df["overview"].apply(remove_emojis)

# Remove Stopwords

In [28]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):

    words = text.split()

    return " ".join(
        word for word in words
        if word not in stop_words
    )

In [29]:
df["overview"]= df["overview"].apply(remove_stopwords)

# Correcting Spellings

In [34]:
spell = SpellChecker()

def spelling_correction(text):
    words = text.split()

    corrected_words = []

    for word in words:
        corrected = spell.correction(word)

        if corrected is not None:
            corrected_words.append(corrected)
        else:
            corrected_words.append(word)

    return " ".join(corrected_words)

In [35]:
from tqdm.auto import tqdm

tqdm.pandas()

df["overview"] = df["overview"].progress_apply(spelling_correction)

100%|██████████| 9980/9980 [1:08:10<00:00,  2.44it/s]


# Apply Tokenization

In [36]:
def tokenize(text):
    return word_tokenize(text)

In [37]:
df["overview"]= df["overview"].apply(tokenize)

In [38]:
df["overview"]

0       [egyptian, desert, deep, polar, ice, caps, eli...
1       [beautiful, taken, boyfriend, rene, bizarre, r...
2       [fifteen, years, original, woodsboro, murders,...
3       [male, young, woman, dark, past, drawn, fate, ...
4       [lucia, natasha, kinky, volatile, excitable, y...
                              ...                        
9975    [beneath, anna, poliatovas, striking, beauty, ...
9976    [fresh, was, vegas, connections, naomi, malone...
9977    [twenty, years, away, odysseus, washes, shores...
9978    [madeline, married, ernest, archrival, helen, ...
9979    [tenacious, attorney, uncovers, dark, secret, ...
Name: overview, Length: 9980, dtype: object

# Applying Stemming

In [39]:
stemmer = PorterStemmer()

def stemming(tokens):

    return [
        stemmer.stem(word)
        for word in tokens
    ]

In [40]:
df["overview"]= df["overview"].apply(stemming)

# Applying Lemmatization

In [41]:
lemmatizer = WordNetLemmatizer()

def lemmatization(tokens):

    return [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

In [42]:
df["overview"]= df["overview"].apply(lemmatization)

In [43]:
df["overview"]

0       [egyptian, desert, deep, polar, ice, cap, elit...
1       [beauti, taken, boyfriend, rene, bizarr, retre...
2       [fifteen, year, origin, woodsboro, murder, sid...
3       [male, young, woman, dark, past, drawn, fate, ...
4       [lucia, natasha, kinki, volatil, excit, young,...
                              ...                        
9975    [beneath, anna, poliatova, strike, beauti, lie...
9976    [fresh, wa, vega, connect, naomi, malon, take,...
9977    [twenti, year, away, odysseu, wash, shore, ith...
9978    [madelin, marri, ernest, archriv, helen, fianc...
9979    [tenaci, attorney, uncov, dark, secret, connec...
Name: overview, Length: 9980, dtype: object

# Now applying more advanced text representation

# One-Hot-Encoding

In [44]:
text= df["overview"]

In [46]:
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

# Your already-tokenized overview column
sentences = df["overview"]

# One-Hot Encoder
mlb = MultiLabelBinarizer()

# Perform encoding
one_hot = mlb.fit_transform(sentences)

# Convert to DataFrame
one_hot_df = pd.DataFrame(
    one_hot,
    columns=mlb.classes_,
    index=df.index
)

# Display
print("Shape:", one_hot_df.shape)

one_hot_df.head()

Shape: (9980, 8050)


,'d,'ll,'re,'s,'ve,00,007,1,10,100,...,zookeep,zorro,zuckerberg,zé,–,—,‘,’,“,”
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [47]:
print("Number of unique words:", len(mlb.classes_))
print(mlb.classes_)

Number of unique words: 8050
["'d" "'ll" "'re" ... '’' '“' '”']


In [48]:
df["overview"]

0       [egyptian, desert, deep, polar, ice, cap, elit...
1       [beauti, taken, boyfriend, rene, bizarr, retre...
2       [fifteen, year, origin, woodsboro, murder, sid...
3       [male, young, woman, dark, past, drawn, fate, ...
4       [lucia, natasha, kinki, volatil, excit, young,...
                              ...                        
9975    [beneath, anna, poliatova, strike, beauti, lie...
9976    [fresh, wa, vega, connect, naomi, malon, take,...
9977    [twenti, year, away, odysseu, wash, shore, ith...
9978    [madelin, marri, ernest, archriv, helen, fianc...
9979    [tenaci, attorney, uncov, dark, secret, connec...
Name: overview, Length: 9980, dtype: object

In [50]:
print(type(df['overview'].iloc[0]))
print(df['overview'].iloc[0])

<class 'list'>
['egyptian', 'desert', 'deep', 'polar', 'ice', 'cap', 'elit', 'gi', 'joe', 'team', 'use', 'latest', 'nextgener', 'spi', 'militari', 'equip', 'fight', 'corrupt', 'arm', 'dealer', 'destroy', 'grow', 'threat', 'mysteri', 'cobra', 'organ', 'prevent', 'plung', 'world', 'chao']


In [51]:
print(type(df['overview'].iloc[1]))
print(df['overview'].iloc[1])

<class 'list'>
['beauti', 'taken', 'boyfriend', 'rene', 'bizarr', 'retreat', 'train', 'bondag', 'sexual', 'pervers']


In [52]:
for i, tokens in enumerate(df['overview']):
    for token in tokens:
        if len(token) > 50:
            print(i, repr(token))

In [54]:
print(mlb.classes_[:30])

["'d" "'ll" "'re" "'s" "'ve" '00' '007' '1' '10' '100' '1057' '108yearold'
 '10yearold' '118th' '11yearold' '12' '12000' '12yearold' '13' '13yearold'
 '14' '1400' '14thcenturi' '14yearold' '15' '150' '1536' '155'
 '15thcenturi' '15yearold']


In [55]:
print(mlb.classes_[-30:])

['yuma' 'yuri' 'zani' 'zebra' 'zeke' 'zenith' 'zero' 'zeroth' 'zeu'
 'zhengzhou' 'zimmerman' 'zinnia' 'zion' 'zionist' 'zipper' 'zodiac' 'zoe'
 'zombi' 'zone' 'zoo' 'zookeep' 'zorro' 'zuckerberg' 'zé' '–' '—' '‘' '’'
 '“' '”']


In [53]:
all_tokens = [token for tokens in df['overview'] for token in tokens]

long_tokens = sorted(all_tokens, key=len, reverse=True)

for token in long_tokens[:20]:
    print(len(token), repr(token))

28 'notquitesobraveassirlancelot'
23 'technologicallysuperior'
21 'cabbieturnedchauffeur'
19 'forever…changed…and'
17 'soldierturnedhigh'
16 'lowermiddleclass'
16 'parapsychologist'
16 'obsessivecompuls'
16 'carefullycontrol'
16 'europeanamerican'
16 'oldoldoldfashion'
16 'malaysiathailand'
15 'lessthanperfect'
15 'povertystricken'
15 'thirteenyearold'
15 'wheelchairbound'
15 'nonetoofriendli'
15 'extinctionlevel'
15 'africanamerican'
15 'africanamerican'


# Bag of Words

In [57]:
from sklearn.feature_extraction.text import CountVectorizer

# Convert token lists into strings
text_for_bow = text.apply(lambda x: ' '.join(x))

# Create CountVectorizer
cv = CountVectorizer()

# Create BoW matrix
bow_matrix = cv.fit_transform(text_for_bow)

# Convert to DataFrame
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=cv.get_feature_names_out(),
    index=text.index
)

print(bow_df.shape)
bow_df.head()

(9980, 8019)


,00,007,10,100,1057,108yearold,10yearold,118th,11yearold,12,...,zipper,zodiac,zoe,zombi,zone,zoo,zookeep,zorro,zuckerberg,zé
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# N-Grams

In [59]:
# Convert token lists into strings
text_for_bow = text.apply(lambda x: ' '.join(x))

# Bigram Vectorizer
cv_bigram = CountVectorizer(
    ngram_range=(2, 2)
)

# Create bigram matrix
bigram_matrix = cv_bigram.fit_transform(text_for_bow)

# Convert to DataFrame
bigram_df = pd.DataFrame(
    bigram_matrix.toarray(),
    columns=cv_bigram.get_feature_names_out(),
    index=text.index
)

print("Shape:", bigram_df.shape)

bigram_df.head()

Shape: (9980, 36276)


,00 agent,007 battl,007 come,007 fight,10 day,10 month,10 year,100 brain,1057 men,108yearold vampir,...,zone specif,zone struggl,zoo anim,zoo forc,zoo stop,zoo wash,zookeep son,zorro escap,zuckerberg begin,zé sequent
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [60]:
cv_ngram = CountVectorizer(
    ngram_range=(1, 2)
)

ngram_matrix = cv_ngram.fit_transform(text_for_bow)

ngram_df = pd.DataFrame(
    ngram_matrix.toarray(),
    columns=cv_ngram.get_feature_names_out(),
    index=text.index
)

print(ngram_df.shape)
ngram_df.head()

(9980, 44295)


,00,00 agent,007,007 battl,007 come,007 fight,10,10 day,10 month,10 year,...,zoo stop,zoo wash,zookeep,zookeep son,zorro,zorro escap,zuckerberg,zuckerberg begin,zé,zé sequent
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [62]:
bigram_matrix = cv_bigram.fit_transform(text.apply(lambda x: ' '.join(x)))

In [63]:
bigram_matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 258526 stored elements and shape (9980, 36276)>

In [64]:
cv_bigram.get_feature_names_out()

array(['00 agent', '007 battl', '007 come', ..., 'zorro escap',
       'zuckerberg begin', 'zé sequent'], shape=(36276,), dtype=object)

# TF-IDF

In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert list of tokens back to a string
text_for_tfidf = text.apply(lambda x: ' '.join(x))

# TF-IDF
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(text_for_tfidf)

# Convert to DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=text.index
)

print("Shape:", tfidf_df.shape)

tfidf_df.head()

Shape: (9980, 8019)


,00,007,10,100,1057,108yearold,10yearold,118th,11yearold,12,...,zipper,zodiac,zoe,zombi,zone,zoo,zookeep,zorro,zuckerberg,zé
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [67]:
tfidf.fit_transform(text.apply(lambda x: ' '.join(x)))

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 252879 stored elements and shape (9980, 8019)>

In [70]:
df.to_csv(r"C:\Users\himan\Education1\Projects\NLP\data\Cleaned_movie_rating.csv")